# İlişkiler ve Sıralamalar

**Modüller:** `pytop.relations`  
**Konu:** İkili ilişkiler, özellik testleri, sıralama türleri, denklik ilişkileri, bölüm kümeleri

---

Bu notebook altı bölümden oluşur:
1. **Konu** — İkili ilişki nedir, temel özellikler
2. **Teoremler** — Denklik sınıfları, bölüm-denklik dualitesi (tam ispat)
3. **Fonksiyon Analizi** — Pseudo-kod, karmaşıklık, avantaj/dezavantaj
4. **API** — `make_relation`, özellik testleri, `total_order_from_list`, `equivalence_from_classes`
5. **Örnekler** — Dört senaryo: kısmi sıralamadan bölüm kümesine
6. **Alıştırmalar** — Kodlama, teori ve karşılaştırma

---
## 1. Konu

### 1.1 İkili İlişki

Bir $X$ kümesi üzerinde **ikili ilişki** $R \subseteq X \times X$'tir.  
$(x, y) \in R$ ise $x \mathrel{R} y$ yazılır.

### 1.2 Temel Özellikler

| Özellik | Tanım | Örnek |
|---------|-------|-------|
| **Yansımalı** (reflexive) | $\forall x: x \mathrel{R} x$ | $\leq$ üzerinde $\mathbb{R}$ |
| **Yansımasız** (irreflexive) | $\forall x: \neg(x \mathrel{R} x)$ | $<$ üzerinde $\mathbb{R}$ |
| **Simetrik** (symmetric) | $x \mathrel{R} y \Rightarrow y \mathrel{R} x$ | arkadaşlık |
| **Antisimetrik** (antisymmetric) | $x \mathrel{R} y$ ve $y \mathrel{R} x \Rightarrow x = y$ | $\leq$ |
| **Geçişli** (transitive) | $x \mathrel{R} y$ ve $y \mathrel{R} z \Rightarrow x \mathrel{R} z$ | $\leq$, $<$ |

### 1.3 Özel İlişki Türleri

| Tür | Özellikler |
|-----|------------|
| **Önkoşul** (preorder) | yansımalı + geçişli |
| **Kısmi sıralama** (partial order) | yansımalı + antisimetrik + geçişli |
| **Tam sıralama** (total/linear order) | kısmi sıralama + her iki eleman karşılaştırılabilir |
| **Denklik ilişkisi** (equivalence) | yansımalı + simetrik + geçişli |

### 1.4 Bölüm Kümeleri

$R$ bir denklik ilişkisi ise her $x \in X$ için **denklik sınıfı** $[x]_R = \{y \in X : x \mathrel{R} y\}$.  
**Bölüm kümesi** (quotient set) $X / R = \{[x]_R : x \in X\}$'dir.

---
## 2. Teoremler

### Teorem 1 — Denklik Sınıfları Bir Bölüm Oluşturur

**İfade:**  
$R$ bir denklik ilişkisi ise $X/R$, $X$'i tam ve ayrık olarak örter (bölümdür):
1. Her $[x]_R \neq \emptyset$
2. $[x]_R = [y]_R$ veya $[x]_R \cap [y]_R = \emptyset$
3. $\bigcup_{x \in X} [x]_R = X$

**İspat:**
1. $x \in [x]_R$ çünkü $R$ yansımalıdır.
2. $[x]_R \cap [y]_R \neq \emptyset$ olsun; $z$ ortak eleman. $x \mathrel{R} z$ ve $y \mathrel{R} z$. Simetri: $z \mathrel{R} y$. Geçişlilik: $x \mathrel{R} y$. Dolayısıyla $[x]_R = [y]_R$.
3. Her $x$, kendi sınıfında: $x \in [x]_R \subseteq \bigcup [x]_R$. $\square$

---

### Teorem 2 — Bölüm-Denklik Dualitesi

**İfade:**  
$X$ üzerindeki denklik ilişkileri ile $X$'in bölümleri birebir örtüşür:
- $R \mapsto X/R$ (ilişkiden bölüm)
- $\Pi \mapsto R_\Pi = \{(x,y) : \exists B \in \Pi,\, x,y \in B\}$ (bölümden ilişki)

**İspat kısa özeti:**  
$R_{X/R} = R$ ve $X/(R_\Pi) = \Pi$ — her iki yön geri adım dönüşüm (retraction) oluşturur. $\square$

---

### Teorem 3 — Ters İlişki Özellikleri

**İfade:**  
$R^{-1} = \{(y, x) : (x, y) \in R\}$ için:
- $R$ yansımalı $\Rightarrow$ $R^{-1}$ yansımalı
- $R$ simetrik $\Rightarrow$ $R^{-1} = R$
- $R$ geçişli $\Rightarrow$ $R^{-1}$ geçişli
- $R$ kısmi sıralama $\Rightarrow$ $R^{-1}$ kısmi sıralama (ters sıralama)

**İspat (geçişli):**  
$(x, y), (y, z) \in R^{-1}$ $\Rightarrow$ $(y, x), (z, y) \in R$ $\Rightarrow$ $(z, x) \in R$ (geçişlilik) $\Rightarrow$ $(x, z) \in R^{-1}$. $\square$

---
## 3. Fonksiyon Analizi

Aşağıda `pytop.relations` modülünün gerçek hesaplama yapan fonksiyonları incelenmektedir.

### `is_reflexive(carrier, R)` / `is_symmetric(carrier, R)` / `is_antisymmetric(carrier, R)`

**Pseudo-kod (`is_reflexive`):**
1. `carrier`'ın her elemanı `x` için:
   - $(x, x) \notin R$ ise → `False` döndür
2. `True` döndür

**Karmaşıklık:**
- Zaman: O(n) — `n = |carrier|`; her eleman için O(1) üyelik kontrolü (set lookup)
- Alan: O(1)

**Avantajlar:**
- Set tabanlı üyelik testi O(1) hash lookup ile gerçekleşir
- Erken çıkış: ilk ihlalde durur

**Dezavantajlar / Sınırlamalar:**
- `R` bir `set` veya `frozenset` olmalı; liste ise O(|R|) lineer arama olur

**Nasıl aşılır:**
- `R`'yi `set` olarak tutmak O(1) üyelik testini garanti eder

---

### `is_transitive(carrier, R)`

**Pseudo-kod:**
1. `R`'nin her `(x, y)` çifti için:
   2. `R`'nin her `(y', z)` çifti için `y' == y`:
      - $(x, z) \notin R$ ise → `False` döndür
3. `True` döndür

**Karmaşıklık:**
- Zaman: O(|R|²) — her çift için R'nin aranması; en kötü O(n⁴) tam çarpım durumunda
- Alan: O(1)

**Avantajlar:**
- Yoğun olmayan (seyrek) ilişkilerde pratikte çok daha hızlı

**Dezavantajlar / Sınırlamalar:**
- Yoğun ilişkilerde yavaş; Warshall algoritması O(n³) ile daha verimli

**Nasıl aşılır:**
- Geçişli kapanım için `pytop.random_generators.random_transitive_relation` Warshall kullanır

---

### `equivalence_class(carrier, R, x)` / `quotient_set(carrier, R)`

**Pseudo-kod (`equivalence_class`):**
1. `carrier`'ın her elemanı `y` için:
   - $(x, y) \in R$ ise `y`'yi sınıfa ekle
2. Sınıfı döndür

**Karmaşıklık:**
- Zaman: O(|R|) — R üzerinde doğrusal tarama
- Alan: O(n) — sınıf elemanları

**`quotient_set`:** O(n · |R|) — her eleman için `equivalence_class` çağrısı, yinelenenler çıkarılır

In [ ]:
from pytop.relations import (
    make_relation, is_reflexive, is_transitive,
    is_partial_order, equivalence_class, quotient_set,
    equivalence_from_classes,
)

carrier = list(range(1, 6))  # {1,2,3,4,5}

# Yansımalılık: O(n) köşegen testi
total_order = make_relation(carrier, *[(x, y) for x in carrier for y in carrier if x <= y])
print(f"Yansımalı (≤): {is_reflexive(carrier, total_order)}")  # True

# Geçişlilik: yoğunluğa göre maliyet farklı
# Seyrek ilişki: sadece (1,2) ve (2,3)
sparse_R = make_relation(carrier, (1, 2), (2, 3))
print(f"Seyrek R geçişli: {is_transitive(carrier, sparse_R)}")   # False: (1,3) eksik

# Denklik sınıfları ve bölüm kümesi
eq_R = equivalence_from_classes([1, 3, 5], [2, 4])
eq_cls = equivalence_class(carrier, eq_R, 1)
quot   = quotient_set(carrier, eq_R)
print(f"[1] denklik sınıfı: {sorted(eq_cls)}")
print(f"X/R = {[sorted(b) for b in quot]}")

---
## 4. API

In [ ]:
from pytop.relations import (
    make_relation,
    total_order_from_list,
    equivalence_from_classes,
    inverse_relation,
    is_reflexive, is_symmetric, is_transitive,
    is_antisymmetric,
    is_partial_order, is_total_order, is_equivalence_relation,
    equivalence_class,
    quotient_set, canonical_projection_from_equivalence,
    relation_profile,
)
print("pytop yüklendi.")

### İlişki Kurucuları

| Fonksiyon | Sözdizimi | Anlamı |
|-----------|-----------|--------|
| `make_relation(taşıyıcı, *çiftler)` | `make_relation([1,2,3], (1,2), (2,3))` | Doğrulanmış ilişki |
| `total_order_from_list(*elemanlar)` | `total_order_from_list(1, 2, 3)` | Tam sıralama |
| `equivalence_from_classes(*bloklar)` | `equivalence_from_classes([1,2],[3])` | Denklik ilişkisi |
| `identity_relation(taşıyıcı)` | `identity_relation([1,2,3])` | Köşegen $\{(x,x)\}$ |
| `inverse_relation(R)` | `inverse_relation(R)` | $R^{-1}$ |
| `compose_relations(R, S)` | `compose_relations(R, S)` | $S \circ R$ |

### Özellik Testleri

| Fonksiyon | Döndürür |
|-----------|----------|
| `is_reflexive(carrier, R)` | `bool` |
| `is_symmetric(carrier, R)` | `bool` |
| `is_transitive(carrier, R)` | `bool` |
| `is_antisymmetric(carrier, R)` | `bool` |
| `is_partial_order(carrier, R)` | `bool` |
| `is_total_order(carrier, R)` | `bool` |
| `is_equivalence_relation(carrier, R)` | `bool` |
| `relation_profile(carrier, R)` | `dict` (tam profil) |

---
## 5. Örnekler

### Örnek 1 — Minimal: İlişki Kurma ve Özellik Testi

In [ ]:
# Taşıyıcı: {1, 2, 3}
carrier = [1, 2, 3]

# Yansımalı + geçişli ama simetrik değil: kısmi sıralama?
R = make_relation(carrier, (1,1), (2,2), (3,3), (1,2), (2,3), (1,3))
print(f"R = {R}")
print(f"Yansımalı:    {is_reflexive(carrier, R)}")
print(f"Simetrik:     {is_symmetric(carrier, R)}")
print(f"Geçişli:      {is_transitive(carrier, R)}")
print(f"Antisimetrik: {is_antisymmetric(carrier, R)}")
print(f"Kısmi sıralama: {is_partial_order(carrier, R)}")
print(f"Tam sıralama:   {is_total_order(carrier, R)}")

### Örnek 2 — Orta Düzey: Tam Sıralama Kurma

In [ ]:
# Sözlüksel sıra: a < b < c < d
carrier2 = ['a', 'b', 'c', 'd']
total_R = total_order_from_list(*carrier2)

print(f"Toplam çift sayısı: {len(total_R)} (beklenen: n(n+1)/2 = 10)")
print(f"Tam sıralama mı? {is_total_order(carrier2, total_R)}")

# Ters sıralama
inv_R = inverse_relation(total_R)
print(f"\nTers ilişki de tam sıralama mı? {is_total_order(carrier2, inv_R)}")
print(f"'d' ≤ 'a' (ters sırada)? {('d', 'a') in inv_R}")

### Örnek 3 — Gelişmiş: Denklik İlişkisi ve Bölüm Kümesi

In [ ]:
# mod 3 denkliği: {0,3,6}, {1,4,7}, {2,5}
universe = list(range(8))   # {0,1,2,3,4,5,6,7}
eq_R = equivalence_from_classes(
    [0, 3, 6],
    [1, 4, 7],
    [2, 5],
)

print(f"Denklik ilişkisi mi? {is_equivalence_relation(universe, eq_R)}")

# Denklik sınıfları
for rep in [0, 1, 2]:
    cls = equivalence_class(universe, eq_R, rep)
    print(f"  [{rep}] = {sorted(cls)}")

# Bölüm kümesi X/R
quot = quotient_set(universe, eq_R)
print(f"\nX/R = {[sorted(b) for b in quot]}")

# Kanonik projeksiyon x ↦ [x]
pi = canonical_projection_from_equivalence(universe, eq_R)
print(f"\nπ(5) = {sorted(pi[5])}   # 5'in denklik sınıfı")

### Örnek 4 — Gerçekçi Senaryo: `relation_profile` ile Tam Analiz

In [ ]:
# Bölünebilirlik ilişkisi: a | b (a, b'yi böler)
divs = [1, 2, 3, 4, 6, 12]
divides_R = make_relation(
    divs,
    *[(a, b) for a in divs for b in divs if b % a == 0]
)

profile = relation_profile(divs, divides_R)
print(f"Taşıyıcı:       {profile['carrier']}")
print(f"İlişki boyutu:  {profile['relation_size']}")
print(f"Yansımalı:      {profile['is_reflexive']}")
print(f"Simetrik:       {profile['is_symmetric']}")
print(f"Antisimetrik:   {profile['is_antisymmetric']}")
print(f"Geçişli:        {profile['is_transitive']}")
print(f"Kısmi sıralama: {profile['is_partial_order']}")
print(f"Tam sıralama:   {profile['is_total_order']}")
print(f"Karşılaştırılamayan çiftler: {len(profile['incomparable_pairs'])}")

---
## 6. Alıştırmalar

### Alıştırma 1 — Kodlama Görevi

$X = \{0, 1, 2, 3, 4\}$ üzerinde mod 2 denkliği tanımla (çiftler ve tekler).  
- `equivalence_from_classes` ile ilişkiyi kur
- Bölüm kümesini yaz
- Her eleman için kanonik projeksiyonu hesapla

In [ ]:
X5 = list(range(5))
eq_mod2 = equivalence_from_classes([0, 2, 4], [1, 3])
print(f"Denklik mi? {is_equivalence_relation(X5, eq_mod2)}")
print(f"X/R = {[sorted(b) for b in quotient_set(X5, eq_mod2)]}")

### Alıştırma 2 — Teori Sorusu

a) Her tam sıralama aynı zamanda kısmi sıralama mıdır? Tersi doğru mu? Bir karşıt örnek ver.  
b) $R = \{(1,2), (2,3)\}$ ilişkisi $\{1,2,3\}$ üzerinde geçişli midir? `is_transitive` ile kontrol et.  
c) Bölünebilirlik ilişkisi neden tam sıralama değildir? Karşılaştırılamayan bir çift bul.

_Yanıtlarınızı buraya yazın:_

a) ...

b) ...

c) ...

### Alıştırma 3 — Karşılaştırma

İki farklı denklik ilişkisi: mod 2 ve mod 3 denkliği.  
- Her birinin bölüm kümesini hesapla
- Bölüm kümelerinin boyutunu karşılaştır
- Hangi denklik daha "kaba" (coarser)?  
  (İpucu: daha az sınıf → daha kaba)

In [ ]:
X6 = list(range(6))
mod2 = equivalence_from_classes([0, 2, 4], [1, 3, 5])
mod3 = equivalence_from_classes([0, 3], [1, 4], [2, 5])

q2 = quotient_set(X6, mod2)
q3 = quotient_set(X6, mod3)
print(f"mod 2: {len(q2)} sınıf → {[sorted(b) for b in q2]}")
print(f"mod 3: {len(q3)} sınıf → {[sorted(b) for b in q3]}")
print(f"Daha kaba olan: {'mod 2' if len(q2) < len(q3) else 'mod 3'}")

### Alıştırma 4 — Karmaşıklık Analizi

a) `is_transitive` fonksiyonu, bir ilişkinin geçişli olup olmadığını O(|R|²) ile kontrol eder. Tam çarpım ilişkisinde (her çift dahil) $n = 5$ için kaç adım gerekir? `relation_profile` ile `is_transitive` sonucunu doğrulayın.

b) `quotient_set`, her eleman için `equivalence_class` çağırır. $|X| = n$ ve $|R| = k$ ise toplam karmaşıklık nedir? $n = 8$, mod-4 denkliğiyle bunu ölçün.

c) `total_order_from_list` ve `inverse_relation` fonksiyonları birleştirildiğinde sonuç yine bir tam sıralama mı? `relation_profile` ile doğrulayın.

In [ ]:
from pytop.relations import (
    make_relation, relation_profile, quotient_set,
    total_order_from_list, inverse_relation, is_total_order,
    equivalence_from_classes,
)

# a) Tam çarpım ilişkisi: n=5, |R|=25
n = 5
carrier5 = list(range(1, n + 1))
full_R = make_relation(carrier5, *[(x, y) for x in carrier5 for y in carrier5])
profile5 = relation_profile(carrier5, full_R)
print(f"Tam çarpım |R|={len(full_R)}, geçişli={profile5['is_transitive']}")
print(f"Adım tahmini: |R|² = {len(full_R)**2}")

# b) quotient_set maliyeti: mod-4 denkliği, n=8
X8 = list(range(8))
mod4 = equivalence_from_classes([0,4],[1,5],[2,6],[3,7])
quot = quotient_set(X8, mod4)
print(f"\nmod-4 bölüm kümesi: {[sorted(b) for b in quot]}")
print(f"|X|=8, |R|={len(mod4)}, bölüm boyutu={len(quot)}")

# c) total_order + inverse = tam sıralama mı?
to = total_order_from_list(1, 2, 3, 4, 5)
inv_to = inverse_relation(to)
print(f"\nTers tam sıralama mı? {is_total_order(carrier5, inv_to)}")

---
## Özet

| Kavram | Ana Sonuç |
|--------|-----------|
| İkili ilişki | $R \subseteq X \times X$; özellikler: yansımalı, simetrik, geçişli, antisimetrik |
| `is_reflexive` | O(n) — köşegen testi, erken çıkış |
| `is_transitive` | O(\|R\|²) — çift çift arama; yoğun ilişkilerde yavaş |
| Kısmi sıralama | Yansımalı + antisimetrik + geçişli; `is_partial_order` tüm üçünü kontrol eder |
| Denklik ilişkisi | Yansımalı + simetrik + geçişli; bölüm kümesiyle birebir örtüşür |
| `equivalence_class` | O(\|R\|) — R'yi filtrele; `quotient_set` O(n · \|R\|) |
| Bölüm-denklik dualitesi | Her denklik ↔ bir bölüm; `equivalence_from_classes` tersten kurar |

**Bir sonraki adım:** `pytop.predicate_sets` — sonsuz matematiksel yapılar ve yüklem tabanlı kümeler.